# SELFIES Variational Autoencoder (VAE) Training

Variational Autoencoder for molecular generation using SELFIES representation.

## Two Training Modes:
1. **Unconditional VAE** - Learn latent space of drug-like molecules
2. **Conditional VAE (CVAE)** - Generate molecules with specific properties

## Architecture:
- **Encoder**: Bidirectional GRU → μ, σ latent parameters
- **Decoder**: GRU with attention → token-by-token generation
- **Loss**: Reconstruction (CE) + KL divergence with annealing

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR

import selfies as sf
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, Draw

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================================
# GOOGLE COLAB SETUP - Run this cell if using Colab
# ============================================================================

# Detect if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("🔵 Running on Google Colab")
except:
    IN_COLAB = False
    print("🟢 Running locally")

if IN_COLAB:
    # ========== OPTION 1: Mount Google Drive (Recommended) ==========
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set your Google Drive path where you uploaded the data
    # Update this path to match your Google Drive structure
    DRIVE_PATH = "/content/drive/MyDrive/Drug_Discovery_Project"
    
    # Check if data exists
    import os
    if os.path.exists(f"{DRIVE_PATH}/data/molecules_200k_selfies_clean.csv"):
        print(f"✅ Found data in Google Drive: {DRIVE_PATH}")
        # Update working directory
        os.chdir(DRIVE_PATH)
    else:
        print(f"❌ Data not found at {DRIVE_PATH}")
        print("Please upload your data folder to Google Drive and update DRIVE_PATH")
    
    # ========== OPTION 2: Direct Upload (Alternative) ==========
    # Uncomment below if you prefer to upload files directly
    # from google.colab import files
    # print("Upload molecules_200k_selfies_clean.csv:")
    # uploaded = files.upload()
    # print("Upload selfies_tokenizer.json:")
    # uploaded = files.upload()
    
    # Install required packages
    print("\n📦 Installing packages...")
    !pip install -q selfies rdkit
    print("✅ Packages installed")

## Setup for Google Colab

**Option 1: Mount Google Drive (Recommended)**
- Upload your `data/` folder to Google Drive
- Run the cell below to mount and access it

**Option 2: Direct Upload**
- Uncomment the upload code below
- Files will be uploaded to Colab's temporary storage (lost after session ends)

## Hyperparameters

In [ ]:
# ============================================================================
# HYPERPARAMETERS
# ============================================================================

class Config:
    # Data Paths - Compatible with both Windows and Google Colab
    # Using string paths that work cross-platform
    DATA_PATH = "data/molecules_200k_selfies_clean.csv"
    TOKENIZER_PATH = "data/selfies_tokenizer.json"
    MODEL_DIR = "../models"
    
    # Sequence
    MAX_LEN = 96              # Maximum sequence length
    
    # ==================== VAE Architecture ====================
    # Encoder
    EMBED_DIM = 128           # Token embedding dimension
    ENCODER_HIDDEN = 512      # GRU hidden size
    ENCODER_LAYERS = 2        # Number of GRU layers
    ENCODER_BIDIRECTIONAL = True  # Use bidirectional GRU
    
    # Latent Space
    LATENT_DIM = 256          # Latent vector dimension
    
    # Decoder
    DECODER_HIDDEN = 512      # GRU hidden size
    DECODER_LAYERS = 3        # Number of GRU layers
    
    # Regularization
    DROPOUT = 0.2             # Dropout rate
    
    # ==================== KL Annealing ====================
    KL_ANNEAL_TYPE = "cyclical"  # "monotonic", "cyclical", or "constant"
    KL_WEIGHT_START = 0.0     # Starting KL weight
    KL_WEIGHT_END = 1.0       # Ending KL weight
    KL_ANNEAL_EPOCHS = 10     # Epochs to anneal over (for monotonic)
    KL_CYCLE_EPOCHS = 4       # Cycle length for cyclical annealing
    
    # ==================== Training ====================
    BATCH_SIZE = 128          # Batch size
    LEARNING_RATE = 3e-4      # Learning rate
    WEIGHT_DECAY = 1e-5       # L2 regularization
    EPOCHS = 50               # Number of training epochs
    GRAD_CLIP = 5.0           # Gradient clipping
    TEACHER_FORCING = 0.9     # Teacher forcing ratio (decay over epochs)
    TF_DECAY = 0.01           # Teacher forcing decay per epoch
    
    # ==================== Conditional Features ====================
    CONDITION_FEATURES = [
        'qed',              # Drug-likeness (0-1)
        'sa_score',         # Synthetic accessibility (1-10)
        'mw',               # Molecular weight
        'alogp',            # Lipophilicity
        'hbd',              # H-bond donors
        'hba',              # H-bond acceptors
        'psa',              # Polar surface area
        'num_rings',        # Number of rings
        'pchembl_mean',     # Binding affinity
    ]
    CONDITION_HIDDEN = 64     # Condition embedding dimension
    
    # ==================== Sampling ====================
    SAMPLE_TEMP = 1.0         # Sampling temperature
    TOP_K = 0                 # Top-k sampling (0 = disabled)
    TOP_P = 0.9               # Nucleus sampling threshold
    
    # ==================== Logging ====================
    LOG_EVERY = 100           # Log every N batches
    SAVE_EVERY = 5            # Save every N epochs
    EVAL_SAMPLES = 1000       # Samples for evaluation


config = Config()

# Create model directory
os.makedirs(config.MODEL_DIR, exist_ok=True)

print("=" * 60)
print("HYPERPARAMETERS")
print("=" * 60)
for key, value in vars(Config).items():
    if not key.startswith('_'):
        print(f"{key}: {value}")

## Load Data & Tokenizer

In [ ]:
# Load dataset
print(f"Loading data from {config.DATA_PATH}...")
df = pd.read_csv(config.DATA_PATH)
print(f"Loaded {len(df):,} molecules")

# Load tokenizer
print(f"\nLoading tokenizer from {config.TOKENIZER_PATH}...")
with open(config.TOKENIZER_PATH, 'r') as f:
    tokenizer_data = json.load(f)

vocab = tokenizer_data['vocab']
stoi = tokenizer_data['stoi']
itos = {int(k): v for k, v in tokenizer_data['itos'].items()}
VOCAB_SIZE = len(vocab)

PAD = tokenizer_data['special_tokens']['PAD']
BOS = tokenizer_data['special_tokens']['BOS']
EOS = tokenizer_data['special_tokens']['EOS']

PAD_IDX = stoi[PAD]
BOS_IDX = stoi[BOS]
EOS_IDX = stoi[EOS]

print(f"\nVocabulary size: {VOCAB_SIZE}")
print(f"Special tokens: PAD={PAD_IDX}, BOS={BOS_IDX}, EOS={EOS_IDX}")

# Show columns
print(f"\nDataset columns ({len(df.columns)}):")
print(df.columns.tolist()[:20], "...")

In [ ]:
# Check conditioning features
print("\n" + "="*60)
print("CONDITIONING FEATURES ANALYSIS")
print("="*60)

available_features = []
for feat in config.CONDITION_FEATURES:
    if feat in df.columns:
        non_null = df[feat].notna().sum()
        coverage = non_null / len(df) * 100
        mean_val = df[feat].mean() if coverage > 0 else 0
        std_val = df[feat].std() if coverage > 0 else 0
        print(f"✅ {feat:15s}: {coverage:5.1f}% coverage | mean={mean_val:8.2f} | std={std_val:8.2f}")
        if coverage > 50:
            available_features.append(feat)
    else:
        print(f"❌ {feat:15s}: NOT FOUND")

print(f"\n🎯 Using {len(available_features)} features for CVAE: {available_features}")

## Tokenization Functions

In [ ]:
def encode_selfies(selfies_str, max_len=config.MAX_LEN):
    """Encode SELFIES to token IDs with BOS/EOS/PAD."""
    tokens = list(sf.split_selfies(selfies_str))
    tokens = [BOS] + tokens + [EOS]
    
    if len(tokens) > max_len:
        tokens = tokens[:max_len]
    
    pad_len = max_len - len(tokens)
    tokens += [PAD] * pad_len
    
    return [stoi[t] for t in tokens]


def decode_tokens(ids, stop_at_eos=True):
    """Decode token IDs to SELFIES string."""
    tokens = []
    for i in ids:
        tok = itos[i]
        if tok == EOS and stop_at_eos:
            break
        if tok not in (PAD, BOS, EOS):
            tokens.append(tok)
    return "".join(tokens)


def selfies_to_smiles(selfies_str):
    """Convert SELFIES to SMILES."""
    try:
        return sf.decoder(selfies_str)
    except:
        return None


def is_valid_molecule(smiles):
    """Check if SMILES is valid."""
    if smiles is None:
        return False
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None


# Test tokenization
test_selfies = df.iloc[0]['selfies']
encoded = encode_selfies(test_selfies)
decoded = decode_tokens(encoded)
print(f"Original: {test_selfies[:60]}...")
print(f"Encoded:  {encoded[:15]}...")
print(f"Decoded:  {decoded[:60]}...")
print(f"Match: {test_selfies == decoded}")

## PyTorch Dataset

In [ ]:
class MoleculeDataset(Dataset):
    """
    PyTorch Dataset for SELFIES molecules.
    Returns (input_tokens, target_tokens, length) for VAE training.
    """
    def __init__(self, df, condition_features=None, max_len=config.MAX_LEN):
        self.selfies = df['selfies'].values
        self.max_len = max_len
        self.condition_features = condition_features
        
        # Precompute token sequences
        print("Encoding sequences...")
        self.sequences = []
        self.lengths = []
        for s in tqdm(self.selfies, desc="Tokenizing"):
            tokens = encode_selfies(s, max_len)
            # Find actual length (up to EOS)
            try:
                length = tokens.index(EOS_IDX) + 1
            except ValueError:
                length = max_len
            self.sequences.append(tokens)
            self.lengths.append(length)
        
        # Prepare conditioning
        if condition_features:
            self.conditions = df[condition_features].values.astype(np.float32)
            # Z-score normalization
            self.cond_mean = np.nanmean(self.conditions, axis=0)
            self.cond_std = np.nanstd(self.conditions, axis=0) + 1e-8
            self.conditions = (self.conditions - self.cond_mean) / self.cond_std
            self.conditions = np.nan_to_num(self.conditions, nan=0.0)
            self.has_conditions = ~np.isnan(df[condition_features].values).any(axis=1)
        else:
            self.conditions = None
            self.has_conditions = None
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        tokens = torch.tensor(self.sequences[idx], dtype=torch.long)
        
        # CRITICAL FIX: Clamp length to match input_tokens length (MAX_LEN - 1)
        # Otherwise pack_padded_sequence can crash when length == MAX_LEN
        length = min(self.lengths[idx], self.max_len - 1)
        
        # Input: [BOS, tok1, tok2, ..., tokN]
        # Target: [tok1, tok2, ..., tokN, EOS]
        input_tokens = tokens[:-1]   # All except last (length = MAX_LEN - 1)
        target_tokens = tokens[1:]   # All except first
        
        if self.conditions is not None:
            cond = torch.tensor(self.conditions[idx], dtype=torch.float32)
            mask = torch.tensor(self.has_conditions[idx], dtype=torch.bool)
            return input_tokens, target_tokens, length, cond, mask
        
        return input_tokens, target_tokens, length


# Create train/val split
train_size = int(0.9 * len(df))
train_df = df.iloc[:train_size].reset_index(drop=True)
val_df = df.iloc[train_size:].reset_index(drop=True)

print(f"\nTrain: {len(train_df):,} molecules")
print(f"Val: {len(val_df):,} molecules")

## VAE Model Architecture

In [ ]:
class Encoder(nn.Module):
    """
    Bidirectional GRU Encoder → Latent μ, σ
    """
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        latent_dim,
        n_layers=2,
        bidirectional=True,
        dropout=0.2,
        padding_idx=0
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        self.bidirectional = bidirectional
        self.n_directions = 2 if bidirectional else 1
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0
        )
        
        # Hidden → Latent
        enc_dim = hidden_dim * self.n_directions * n_layers
        self.fc_mu = nn.Linear(enc_dim, latent_dim)
        self.fc_logvar = nn.Linear(enc_dim, latent_dim)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, lengths=None):
        """
        Args:
            x: Token indices [batch, seq_len]
            lengths: Actual sequence lengths [batch]
        Returns:
            mu, logvar: Latent parameters [batch, latent_dim]
        """
        batch_size = x.size(0)
        
        # Embed
        embedded = self.dropout(self.embedding(x))
        
        # Pack if lengths provided
        if lengths is not None:
            lengths = lengths.cpu()
            packed = nn.utils.rnn.pack_padded_sequence(
                embedded, lengths, batch_first=True, enforce_sorted=False
            )
            _, hidden = self.gru(packed)
        else:
            _, hidden = self.gru(embedded)
        
        # hidden: [n_layers * n_directions, batch, hidden_dim]
        # Flatten all layers/directions
        hidden = hidden.permute(1, 0, 2).contiguous()
        hidden = hidden.view(batch_size, -1)
        
        # Latent parameters
        mu = self.fc_mu(hidden)
        logvar = self.fc_logvar(hidden)
        
        return mu, logvar


class Decoder(nn.Module):
    """
    GRU Decoder with latent conditioning.
    """
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        latent_dim,
        n_layers=3,
        dropout=0.2,
        padding_idx=0,
        condition_dim=0
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.n_layers = n_layers
        self.condition_dim = condition_dim
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        
        # Latent + condition → initial hidden
        total_latent = latent_dim + condition_dim
        self.latent_to_hidden = nn.Linear(total_latent, hidden_dim * n_layers)
        
        # GRU input: embedding + latent (concatenated at each step)
        self.gru = nn.GRU(
            embed_dim + total_latent,
            hidden_dim,
            n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        
        # Output projection
        self.output = nn.Linear(hidden_dim, vocab_size)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, z, hidden=None, condition=None):
        """
        Args:
            x: Input tokens [batch, seq_len]
            z: Latent vector [batch, latent_dim]
            hidden: Previous hidden state (optional)
            condition: Condition features [batch, condition_dim] (optional)
        Returns:
            logits: [batch, seq_len, vocab_size]
            hidden: Final hidden state
        """
        batch_size, seq_len = x.size()
        
        # Concatenate z and condition if provided
        if condition is not None:
            z_cond = torch.cat([z, condition], dim=-1)
        else:
            z_cond = z
        
        # Initialize hidden from latent
        if hidden is None:
            hidden = self.latent_to_hidden(z_cond)
            hidden = hidden.view(batch_size, self.n_layers, self.hidden_dim)
            hidden = hidden.permute(1, 0, 2).contiguous()
        
        # Embed tokens
        embedded = self.dropout(self.embedding(x))
        
        # Concatenate z to each timestep
        z_expanded = z_cond.unsqueeze(1).expand(-1, seq_len, -1)
        gru_input = torch.cat([embedded, z_expanded], dim=-1)
        
        # GRU forward
        output, hidden = self.gru(gru_input, hidden)
        
        # Project to vocabulary
        logits = self.output(output)
        
        return logits, hidden


class MoleculeVAE(nn.Module):
    """
    Complete VAE for molecular generation.
    """
    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        encoder_hidden=512,
        encoder_layers=2,
        decoder_hidden=512,
        decoder_layers=3,
        latent_dim=256,
        dropout=0.2,
        bidirectional=True,
        padding_idx=0,
        condition_dim=0,
        n_conditions=0
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.conditional = n_conditions > 0
        
        # Condition embedding
        if self.conditional:
            self.condition_net = nn.Sequential(
                nn.Linear(n_conditions, condition_dim),
                nn.ReLU(),
                nn.Linear(condition_dim, condition_dim)
            )
            self.condition_dim = condition_dim
        else:
            self.condition_dim = 0
        
        self.encoder = Encoder(
            vocab_size, embed_dim, encoder_hidden, latent_dim,
            encoder_layers, bidirectional, dropout, padding_idx
        )
        
        self.decoder = Decoder(
            vocab_size, embed_dim, decoder_hidden, latent_dim,
            decoder_layers, dropout, padding_idx, self.condition_dim
        )
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = μ + σ * ε"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x, lengths=None, condition=None, condition_mask=None):
        """
        Forward pass for training.
        
        Args:
            x: Input tokens [batch, seq_len] (with BOS)
            lengths: Sequence lengths [batch]
            condition: Raw condition features [batch, n_conditions]
            condition_mask: Which samples have valid conditions [batch]
        
        Returns:
            logits, mu, logvar
        """
        # Encode
        mu, logvar = self.encoder(x, lengths)
        
        # Sample latent
        z = self.reparameterize(mu, logvar)
        
        # Process condition
        cond_emb = None
        if self.conditional and condition is not None:
            cond_emb = self.condition_net(condition)
            if condition_mask is not None:
                # Zero out conditions for samples without valid data
                cond_emb = cond_emb * condition_mask.float().unsqueeze(-1)
        
        # Decode
        logits, _ = self.decoder(x, z, condition=cond_emb)
        
        return logits, mu, logvar
    
    def encode(self, x, lengths=None):
        """Encode to latent space."""
        mu, logvar = self.encoder(x, lengths)
        return mu, logvar
    
    def decode(self, z, max_len, condition=None, temperature=1.0, top_k=0, top_p=0.9):
        """
        Decode from latent space (autoregressive generation).
        
        Args:
            z: Latent vectors [batch, latent_dim]
            max_len: Maximum sequence length
            condition: Condition embeddings [batch, condition_dim]
            temperature: Sampling temperature
            top_k: Top-k sampling (0 = disabled)
            top_p: Nucleus sampling threshold
        
        Returns:
            Generated token sequences [batch, max_len]
        """
        batch_size = z.size(0)
        device = z.device
        
        # Start with BOS token
        generated = torch.full((batch_size, 1), BOS_IDX, dtype=torch.long, device=device)
        hidden = None
        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
        
        for _ in range(max_len - 1):
            # Get logits for last token
            logits, hidden = self.decoder(generated[:, -1:], z, hidden, condition)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k > 0:
                indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
                logits[indices_to_remove] = -float('inf')
            
            # Top-p (nucleus) sampling
            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits[indices_to_remove] = -float('inf')
            
            # Sample
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            
            # Update finished status
            finished = finished | (next_token.squeeze(-1) == EOS_IDX)
            
            # Append
            generated = torch.cat([generated, next_token], dim=1)
            
            # Stop if all finished
            if finished.all():
                break
        
        return generated
    
    @torch.no_grad()
    def sample(self, num_samples, max_len, condition=None, temperature=1.0, top_k=0, top_p=0.9, device='cuda'):
        """
        Sample molecules from prior.
        """
        self.eval()
        
        # Sample from prior N(0, 1)
        z = torch.randn(num_samples, self.latent_dim, device=device)
        
        # Process condition if provided
        cond_emb = None
        if self.conditional and condition is not None:
            cond_emb = self.condition_net(condition)
        
        # Generate
        generated = self.decode(z, max_len, cond_emb, temperature, top_k, top_p)
        
        return generated


# Test model
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

test_vae = MoleculeVAE(
    vocab_size=VOCAB_SIZE,
    embed_dim=config.EMBED_DIM,
    encoder_hidden=config.ENCODER_HIDDEN,
    encoder_layers=config.ENCODER_LAYERS,
    decoder_hidden=config.DECODER_HIDDEN,
    decoder_layers=config.DECODER_LAYERS,
    latent_dim=config.LATENT_DIM,
    dropout=config.DROPOUT,
    bidirectional=config.ENCODER_BIDIRECTIONAL,
    padding_idx=PAD_IDX
).to(device)

print(f"\n📊 VAE Model:")
print(f"  Total parameters: {count_parameters(test_vae):,}")
print(f"  Model size: {count_parameters(test_vae) * 4 / 1e6:.1f} MB (fp32)")

# Test forward
x_test = torch.randint(0, VOCAB_SIZE, (4, config.MAX_LEN - 1)).to(device)
logits, mu, logvar = test_vae(x_test)
print(f"\n  Input shape: {x_test.shape}")
print(f"  Output logits: {logits.shape}")
print(f"  Latent μ: {mu.shape}")
print(f"  Latent σ²: {logvar.shape}")

del test_vae

## Loss Function & KL Annealing

In [ ]:
def vae_loss(logits, targets, mu, logvar, kl_weight=1.0):
    """
    VAE ELBO loss = Reconstruction + β * KL Divergence
    
    Args:
        logits: Predicted logits [batch, seq_len, vocab_size]
        targets: Target tokens [batch, seq_len]
        mu, logvar: Latent distribution parameters
        kl_weight: β coefficient for KL term (annealing)
    
    Returns:
        total_loss, recon_loss, kl_loss
    """
    batch_size = logits.size(0)
    
    # Reconstruction loss (cross-entropy)
    recon_loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1),
        ignore_index=PAD_IDX,
        reduction='sum'
    ) / batch_size
    
    # KL divergence: D_KL(q(z|x) || p(z)) where p(z) = N(0, 1)
    # = -0.5 * sum(1 + log(σ²) - μ² - σ²)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size
    
    # Total loss
    total_loss = recon_loss + kl_weight * kl_loss
    
    return total_loss, recon_loss, kl_loss


def get_kl_weight(epoch, batch_idx, num_batches, config):
    """
    Get KL weight based on annealing schedule.
    """
    total_steps = epoch * num_batches + batch_idx
    
    if config.KL_ANNEAL_TYPE == "constant":
        return config.KL_WEIGHT_END
    
    elif config.KL_ANNEAL_TYPE == "monotonic":
        # Linear increase over KL_ANNEAL_EPOCHS
        anneal_steps = config.KL_ANNEAL_EPOCHS * num_batches
        if total_steps >= anneal_steps:
            return config.KL_WEIGHT_END
        ratio = total_steps / anneal_steps
        return config.KL_WEIGHT_START + ratio * (config.KL_WEIGHT_END - config.KL_WEIGHT_START)
    
    elif config.KL_ANNEAL_TYPE == "cyclical":
        # Cyclical annealing (improves training stability)
        cycle_steps = config.KL_CYCLE_EPOCHS * num_batches
        cycle_pos = total_steps % cycle_steps
        ratio = min(1.0, cycle_pos / (cycle_steps * 0.5))  # Ramp up in first half of cycle
        return config.KL_WEIGHT_START + ratio * (config.KL_WEIGHT_END - config.KL_WEIGHT_START)
    
    else:
        return 1.0


@torch.no_grad()
def evaluate_generation_quality(model, num_samples=200, max_len=config.MAX_LEN, 
                                  temperature=1.0, top_p=0.9, condition=None):
    """
    Evaluate generation quality: validity, uniqueness, mean length.
    Detects degeneration like premature EOS spam.
    """
    model.eval()
    
    # Sample from model
    samples = model.sample(
        num_samples=num_samples,
        max_len=max_len,
        condition=condition,
        temperature=temperature,
        top_p=top_p,
        device=device
    )
    
    # Decode and analyze
    lengths = []  # Length until EOS (catches degeneration)
    valid_smiles = []
    all_smiles = []
    
    for seq in samples:
        seq_list = seq.cpu().tolist()
        
        # Find length until EOS (excluding BOS)
        try:
            eos_pos = seq_list.index(EOS_IDX)
            length = eos_pos - 1  # Subtract 1 for BOS
        except ValueError:
            length = len(seq_list) - 1  # No EOS found, full length
        lengths.append(max(0, length))
        
        # Decode to SELFIES/SMILES
        selfies = decode_tokens(seq_list)
        smiles = selfies_to_smiles(selfies)
        all_smiles.append(smiles)
        
        if is_valid_molecule(smiles):
            valid_smiles.append(smiles)
    
    # Compute metrics
    validity = len(valid_smiles) / len(all_smiles) * 100 if all_smiles else 0
    uniqueness = len(set(valid_smiles)) / len(valid_smiles) * 100 if valid_smiles else 0
    mean_length = np.mean(lengths) if lengths else 0
    std_length = np.std(lengths) if lengths else 0
    
    # Degeneration indicators
    short_seqs = sum(1 for l in lengths if l < 5)  # Very short = likely EOS spam
    empty_seqs = sum(1 for l in lengths if l == 0)
    
    return {
        'validity': validity,
        'uniqueness': uniqueness,
        'mean_length': mean_length,
        'std_length': std_length,
        'short_seqs': short_seqs,
        'empty_seqs': empty_seqs,
        'num_samples': num_samples
    }


print("Loss functions defined.")
print(f"KL Annealing type: {config.KL_ANNEAL_TYPE}")

---
# Stage 1: Unconditional VAE Training
---

In [ ]:
# Create datasets (unconditional)
train_dataset_uncond = MoleculeDataset(train_df, condition_features=None)
val_dataset_uncond = MoleculeDataset(val_df, condition_features=None)

train_loader_uncond = DataLoader(
    train_dataset_uncond,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
val_loader_uncond = DataLoader(
    val_dataset_uncond,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"Train batches: {len(train_loader_uncond)}")
print(f"Val batches: {len(val_loader_uncond)}")

In [ ]:
# Create unconditional VAE
vae_uncond = MoleculeVAE(
    vocab_size=VOCAB_SIZE,
    embed_dim=config.EMBED_DIM,
    encoder_hidden=config.ENCODER_HIDDEN,
    encoder_layers=config.ENCODER_LAYERS,
    decoder_hidden=config.DECODER_HIDDEN,
    decoder_layers=config.DECODER_LAYERS,
    latent_dim=config.LATENT_DIM,
    dropout=config.DROPOUT,
    bidirectional=config.ENCODER_BIDIRECTIONAL,
    padding_idx=PAD_IDX,
    condition_dim=0,
    n_conditions=0
).to(device)

print(f"Unconditional VAE parameters: {count_parameters(vae_uncond):,}")

# Optimizer
optimizer_uncond = AdamW(
    vae_uncond.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Scheduler
scheduler_uncond = ReduceLROnPlateau(
    optimizer_uncond, mode='min', factor=0.5, patience=3, verbose=True
)

In [ ]:
def train_vae_epoch(model, loader, optimizer, epoch, config, conditional=False):
    """Train VAE for one epoch."""
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    num_batches = len(loader)
    
    # Teacher forcing ratio (decay over epochs)
    tf_ratio = max(0.5, config.TEACHER_FORCING - epoch * config.TF_DECAY)
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}")
    for batch_idx, batch in enumerate(pbar):
        if conditional:
            input_tokens, target_tokens, lengths, cond, cond_mask = batch
            cond = cond.to(device)
            cond_mask = cond_mask.to(device)
        else:
            input_tokens, target_tokens, lengths = batch
            cond = None
            cond_mask = None
        
        input_tokens = input_tokens.to(device)
        target_tokens = target_tokens.to(device)
        
        # KL annealing weight
        kl_weight = get_kl_weight(epoch, batch_idx, num_batches, config)
        
        # Forward
        logits, mu, logvar = model(input_tokens, lengths, cond, cond_mask)
        
        # Loss
        loss, recon_loss, kl_loss = vae_loss(logits, target_tokens, mu, logvar, kl_weight)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP)
        optimizer.step()
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
        
        if batch_idx % config.LOG_EVERY == 0:
            pbar.set_postfix({
                'loss': f"{loss.item():.2f}",
                'recon': f"{recon_loss.item():.2f}",
                'kl': f"{kl_loss.item():.2f}",
                'β': f"{kl_weight:.3f}"
            })
    
    n = len(loader)
    return total_loss/n, total_recon/n, total_kl/n


@torch.no_grad()
def validate_vae(model, loader, config, conditional=False):
    """Validate VAE."""
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    
    for batch in loader:
        if conditional:
            input_tokens, target_tokens, lengths, cond, cond_mask = batch
            cond = cond.to(device)
            cond_mask = cond_mask.to(device)
        else:
            input_tokens, target_tokens, lengths = batch
            cond = None
            cond_mask = None
        
        input_tokens = input_tokens.to(device)
        target_tokens = target_tokens.to(device)
        
        logits, mu, logvar = model(input_tokens, lengths, cond, cond_mask)
        loss, recon_loss, kl_loss = vae_loss(logits, target_tokens, mu, logvar, kl_weight=1.0)
        
        total_loss += loss.item()
        total_recon += recon_loss.item()
        total_kl += kl_loss.item()
    
    n = len(loader)
    return total_loss/n, total_recon/n, total_kl/n

In [ ]:
# ============================================================================
# TRAIN UNCONDITIONAL VAE
# ============================================================================

print("="*60)
print("STAGE 1: UNCONDITIONAL VAE TRAINING")
print("="*60)
print("\nPer-epoch metrics: recon_loss, kl_loss, validity%, uniqueness%, mean_length")
print("(mean_length < 10 may indicate EOS spam / degeneration)\n")

history_uncond = defaultdict(list)
best_val_loss = float('inf')

for epoch in range(config.EPOCHS):
    # Train
    train_loss, train_recon, train_kl = train_vae_epoch(
        vae_uncond, train_loader_uncond, optimizer_uncond, epoch, config, conditional=False
    )
    
    # Validate
    val_loss, val_recon, val_kl = validate_vae(
        vae_uncond, val_loader_uncond, config, conditional=False
    )
    
    # Evaluate generation quality (every epoch)
    gen_metrics = evaluate_generation_quality(
        vae_uncond, num_samples=200, max_len=config.MAX_LEN,
        temperature=config.SAMPLE_TEMP, top_p=config.TOP_P
    )
    
    # Update scheduler
    scheduler_uncond.step(val_loss)
    
    # Log everything
    history_uncond['train_loss'].append(train_loss)
    history_uncond['train_recon'].append(train_recon)
    history_uncond['train_kl'].append(train_kl)
    history_uncond['val_loss'].append(val_loss)
    history_uncond['val_recon'].append(val_recon)
    history_uncond['val_kl'].append(val_kl)
    history_uncond['validity'].append(gen_metrics['validity'])
    history_uncond['uniqueness'].append(gen_metrics['uniqueness'])
    history_uncond['mean_length'].append(gen_metrics['mean_length'])
    history_uncond['std_length'].append(gen_metrics['std_length'])
    
    # Print comprehensive metrics
    print(f"Epoch {epoch+1}/{config.EPOCHS}")
    print(f"  Loss:  train={train_loss:.3f}, val={val_loss:.3f}")
    print(f"  Recon: train={train_recon:.3f}, val={val_recon:.3f}")
    print(f"  KL:    train={train_kl:.3f}, val={val_kl:.3f}")
    print(f"  Gen:   validity={gen_metrics['validity']:.1f}%, uniqueness={gen_metrics['uniqueness']:.1f}%")
    print(f"  Len:   mean={gen_metrics['mean_length']:.1f} ± {gen_metrics['std_length']:.1f}" + 
          (f" ⚠️ SHORT" if gen_metrics['mean_length'] < 10 else "") +
          (f" ({gen_metrics['empty_seqs']} empty)" if gen_metrics['empty_seqs'] > 0 else ""))
    
    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': vae_uncond.state_dict(),
            'optimizer_state_dict': optimizer_uncond.state_dict(),
            'val_loss': val_loss,
            'config': vars(config),
            'gen_metrics': gen_metrics
        }, os.path.join(config.MODEL_DIR, 'vae_uncond_best.pt'))
        print(f"  ✅ Saved best model")
    
    # Checkpoint
    if (epoch + 1) % config.SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': vae_uncond.state_dict(),
            'history': dict(history_uncond)
        }, os.path.join(config.MODEL_DIR, f'vae_uncond_epoch{epoch+1}.pt'))

print("\n✅ Unconditional VAE training complete!")

## Sample from Unconditional VAE

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(config.MODEL_DIR, 'vae_uncond_best.pt'))
vae_uncond.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']+1}")

# Generate samples
print("\nGenerating 100 molecules...")
samples = vae_uncond.sample(
    num_samples=100,
    max_len=config.MAX_LEN,
    temperature=config.SAMPLE_TEMP,
    top_p=config.TOP_P,
    device=device
)

# Decode and evaluate
generated_selfies = [decode_tokens(s.cpu().tolist()) for s in samples]
generated_smiles = [selfies_to_smiles(s) for s in generated_selfies]
valid_smiles = [s for s in generated_smiles if is_valid_molecule(s)]

validity = len(valid_smiles) / len(generated_smiles) * 100
uniqueness = len(set(valid_smiles)) / len(valid_smiles) * 100 if valid_smiles else 0

print(f"\n=== Unconditional VAE Generation ===")
print(f"Validity: {validity:.1f}%")
print(f"Uniqueness: {uniqueness:.1f}%")

# Show samples with properties
print(f"\nSample molecules:")
for i, smi in enumerate(valid_smiles[:5]):
    mol = Chem.MolFromSmiles(smi)
    qed_score = QED.qed(mol)
    mw = Descriptors.MolWt(mol)
    print(f"  {i+1}. {smi[:50]}... | QED={qed_score:.3f}, MW={mw:.0f}")

---
# Stage 2: Conditional VAE (CVAE) Training
---

In [ ]:
# Create conditional datasets
print(f"Using conditioning features: {available_features}")

train_dataset_cond = MoleculeDataset(train_df, condition_features=available_features)
val_dataset_cond = MoleculeDataset(val_df, condition_features=available_features)

train_loader_cond = DataLoader(
    train_dataset_cond,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
val_loader_cond = DataLoader(
    val_dataset_cond,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Save normalization stats
cond_stats = {
    'features': available_features,
    'mean': train_dataset_cond.cond_mean.tolist(),
    'std': train_dataset_cond.cond_std.tolist()
}
with open(os.path.join(config.MODEL_DIR, 'vae_condition_stats.json'), 'w') as f:
    json.dump(cond_stats, f, indent=2)

print(f"Saved condition stats.")
print(f"Train batches: {len(train_loader_cond)}")

In [ ]:
# Create conditional VAE
vae_cond = MoleculeVAE(
    vocab_size=VOCAB_SIZE,
    embed_dim=config.EMBED_DIM,
    encoder_hidden=config.ENCODER_HIDDEN,
    encoder_layers=config.ENCODER_LAYERS,
    decoder_hidden=config.DECODER_HIDDEN,
    decoder_layers=config.DECODER_LAYERS,
    latent_dim=config.LATENT_DIM,
    dropout=config.DROPOUT,
    bidirectional=config.ENCODER_BIDIRECTIONAL,
    padding_idx=PAD_IDX,
    condition_dim=config.CONDITION_HIDDEN,
    n_conditions=len(available_features)
).to(device)

print(f"Conditional VAE parameters: {count_parameters(vae_cond):,}")

# Optimizer
optimizer_cond = AdamW(
    vae_cond.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

scheduler_cond = ReduceLROnPlateau(
    optimizer_cond, mode='min', factor=0.5, patience=3, verbose=True
)

In [ ]:
# ============================================================================
# TRAIN CONDITIONAL VAE
# ============================================================================

print("="*60)
print("STAGE 2: CONDITIONAL VAE (CVAE) TRAINING")
print("="*60)
print("\nPer-epoch metrics: recon_loss, kl_loss, validity%, uniqueness%, mean_length")
print("(mean_length < 10 may indicate EOS spam / degeneration)\n")

history_cond = defaultdict(list)
best_val_loss_cond = float('inf')

# Prepare a fixed condition for consistent evaluation (high QED target)
eval_condition = torch.zeros(200, len(available_features), device=device)
eval_condition[:, available_features.index('qed')] = (0.8 - train_dataset_cond.cond_mean[available_features.index('qed')]) / train_dataset_cond.cond_std[available_features.index('qed')]

for epoch in range(config.EPOCHS):
    # Train
    train_loss, train_recon, train_kl = train_vae_epoch(
        vae_cond, train_loader_cond, optimizer_cond, epoch, config, conditional=True
    )
    
    # Validate
    val_loss, val_recon, val_kl = validate_vae(
        vae_cond, val_loader_cond, config, conditional=True
    )
    
    # Evaluate generation quality with conditioning
    vae_cond.eval()
    with torch.no_grad():
        cond_emb_eval = vae_cond.condition_net(eval_condition)
    gen_metrics = evaluate_generation_quality(
        vae_cond, num_samples=200, max_len=config.MAX_LEN,
        temperature=config.SAMPLE_TEMP, top_p=config.TOP_P,
        condition=cond_emb_eval
    )
    
    scheduler_cond.step(val_loss)
    
    # Log everything
    history_cond['train_loss'].append(train_loss)
    history_cond['train_recon'].append(train_recon)
    history_cond['train_kl'].append(train_kl)
    history_cond['val_loss'].append(val_loss)
    history_cond['val_recon'].append(val_recon)
    history_cond['val_kl'].append(val_kl)
    history_cond['validity'].append(gen_metrics['validity'])
    history_cond['uniqueness'].append(gen_metrics['uniqueness'])
    history_cond['mean_length'].append(gen_metrics['mean_length'])
    history_cond['std_length'].append(gen_metrics['std_length'])
    
    # Print comprehensive metrics
    print(f"Epoch {epoch+1}/{config.EPOCHS}")
    print(f"  Loss:  train={train_loss:.3f}, val={val_loss:.3f}")
    print(f"  Recon: train={train_recon:.3f}, val={val_recon:.3f}")
    print(f"  KL:    train={train_kl:.3f}, val={val_kl:.3f}")
    print(f"  Gen:   validity={gen_metrics['validity']:.1f}%, uniqueness={gen_metrics['uniqueness']:.1f}%")
    print(f"  Len:   mean={gen_metrics['mean_length']:.1f} ± {gen_metrics['std_length']:.1f}" + 
          (f" ⚠️ SHORT" if gen_metrics['mean_length'] < 10 else "") +
          (f" ({gen_metrics['empty_seqs']} empty)" if gen_metrics['empty_seqs'] > 0 else ""))
    
    # Save best
    if val_loss < best_val_loss_cond:
        best_val_loss_cond = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': vae_cond.state_dict(),
            'optimizer_state_dict': optimizer_cond.state_dict(),
            'val_loss': val_loss,
            'config': vars(config),
            'condition_features': available_features,
            'gen_metrics': gen_metrics
        }, os.path.join(config.MODEL_DIR, 'vae_cond_best.pt'))
        print(f"  ✅ Saved best model")
    
    # Checkpoint
    if (epoch + 1) % config.SAVE_EVERY == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': vae_cond.state_dict(),
            'history': dict(history_cond)
        }, os.path.join(config.MODEL_DIR, f'vae_cond_epoch{epoch+1}.pt'))

print("\n✅ Conditional VAE training complete!")

## Sample from Conditional VAE

In [ ]:
def prepare_conditions(target_values, cond_stats, num_samples, device):
    """Prepare normalized conditions for sampling."""
    features = cond_stats['features']
    mean = np.array(cond_stats['mean'])
    std = np.array(cond_stats['std'])
    
    values = [target_values.get(feat, mean[i]) for i, feat in enumerate(features)]
    values = np.array(values)
    values_norm = (values - mean) / std
    
    return torch.tensor(values_norm, dtype=torch.float32, device=device).unsqueeze(0).expand(num_samples, -1)


# Load best CVAE
checkpoint = torch.load(os.path.join(config.MODEL_DIR, 'vae_cond_best.pt'))
vae_cond.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best CVAE from epoch {checkpoint['epoch']+1}")

# Load condition stats
with open(os.path.join(config.MODEL_DIR, 'vae_condition_stats.json'), 'r') as f:
    cond_stats = json.load(f)

In [ ]:
# Generate HIGH QED molecules
print("\n" + "="*60)
print("Generating HIGH QED molecules (target QED=0.85)")
print("="*60)

num_samples = 100
target_high_qed = {
    'qed': 0.85,
    'sa_score': 3.0,
}

cond_batch = prepare_conditions(target_high_qed, cond_stats, num_samples, device)

vae_cond.eval()
with torch.no_grad():
    z = torch.randn(num_samples, config.LATENT_DIM, device=device)
    cond_emb = vae_cond.condition_net(cond_batch)
    samples = vae_cond.decode(
        z, config.MAX_LEN, cond_emb,
        temperature=config.SAMPLE_TEMP,
        top_p=config.TOP_P
    )

# Evaluate
generated_selfies = [decode_tokens(s.cpu().tolist()) for s in samples]
generated_smiles = [selfies_to_smiles(s) for s in generated_selfies]
valid_smiles = [s for s in generated_smiles if is_valid_molecule(s)]

qed_scores = []
for smi in valid_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        qed_scores.append(QED.qed(mol))

print(f"\nResults:")
print(f"  Validity: {len(valid_smiles)/len(generated_smiles)*100:.1f}%")
print(f"  Mean QED: {np.mean(qed_scores):.3f}" if qed_scores else "N/A")
print(f"  % with QED > 0.7: {sum(q > 0.7 for q in qed_scores)/len(qed_scores)*100:.1f}%" if qed_scores else "N/A")

print(f"\nTop 5 by QED:")
sorted_pairs = sorted(zip(valid_smiles, qed_scores), key=lambda x: x[1], reverse=True)
for i, (smi, qed) in enumerate(sorted_pairs[:5]):
    print(f"  {i+1}. QED={qed:.3f}: {smi[:50]}...")

In [ ]:
# Generate LOW molecular weight molecules
print("\n" + "="*60)
print("Generating LOW MW molecules (target MW=300)")
print("="*60)

target_low_mw = {
    'qed': 0.75,
    'mw': 300,
    'sa_score': 3.0,
}

cond_batch = prepare_conditions(target_low_mw, cond_stats, num_samples, device)

with torch.no_grad():
    z = torch.randn(num_samples, config.LATENT_DIM, device=device)
    cond_emb = vae_cond.condition_net(cond_batch)
    samples = vae_cond.decode(
        z, config.MAX_LEN, cond_emb,
        temperature=config.SAMPLE_TEMP,
        top_p=config.TOP_P
    )

# Evaluate
generated_selfies = [decode_tokens(s.cpu().tolist()) for s in samples]
generated_smiles = [selfies_to_smiles(s) for s in generated_selfies]
valid_smiles = [s for s in generated_smiles if is_valid_molecule(s)]

mw_values = []
for smi in valid_smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        mw_values.append(Descriptors.MolWt(mol))

print(f"\nResults:")
print(f"  Validity: {len(valid_smiles)/len(generated_smiles)*100:.1f}%")
print(f"  Mean MW: {np.mean(mw_values):.0f}" if mw_values else "N/A")
print(f"  % with MW < 350: {sum(m < 350 for m in mw_values)/len(mw_values)*100:.1f}%" if mw_values else "N/A")

print(f"\nTop 5 by lowest MW:")
sorted_pairs = sorted(zip(valid_smiles, mw_values), key=lambda x: x[1])
for i, (smi, mw) in enumerate(sorted_pairs[:5]):
    print(f"  {i+1}. MW={mw:.0f}: {smi[:50]}...")

---
# Stage 3: Affinity-Conditioned VAE (Fine-tune on pchembl_mean subset)
---

Fine-tune the CVAE on molecules with binding affinity data for potency-guided generation.

**Key:** Load encoder/decoder from Stage 2, but re-initialize condition_net (n_conditions changes when including pchembl_mean).

In [ ]:
# ---
# Stage 3: Affinity-Conditioned VAE (Fine-tune on pchembl_mean subset)
# ---

print("\n" + "="*60)
print("STAGE 3: AFFINITY-CONDITIONED CVAE (pchembl_mean subset)")
print("="*60)

# Filter molecules with valid pchembl_mean
df_aff = df[df["pchembl_mean"].notna()].copy().reset_index(drop=True)

# CRITICAL: Shuffle before split to avoid ordering bias
df_aff = df_aff.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Affinity subset size: {len(df_aff):,} / {len(df):,} ({len(df_aff)/len(df)*100:.1f}%)")

# Basic distribution
print("\npchembl_mean stats:")
print(df_aff["pchembl_mean"].describe())

# Train/val split (keep same 90/10)
train_size_aff = int(0.9 * len(df_aff))
train_df_aff = df_aff.iloc[:train_size_aff].reset_index(drop=True)
val_df_aff   = df_aff.iloc[train_size_aff:].reset_index(drop=True)

print(f"\nTrain (aff): {len(train_df_aff):,}")
print(f"Val   (aff): {len(val_df_aff):,}")

In [ ]:
# Stage 3 conditioning features = Stage 2 features + pchembl_mean (explicit)
stage3_features = [f for f in config.CONDITION_FEATURES if f in df.columns]
if "pchembl_mean" not in stage3_features:
    stage3_features.append("pchembl_mean")

print("Stage 3 conditioning features:")
print(stage3_features)

In [ ]:
# Create affinity-conditioned datasets
train_dataset_aff = MoleculeDataset(train_df_aff, condition_features=stage3_features)
val_dataset_aff   = MoleculeDataset(val_df_aff, condition_features=stage3_features)

train_loader_aff = DataLoader(
    train_dataset_aff,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)
val_loader_aff = DataLoader(
    val_dataset_aff,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

# Save normalization stats for Stage 3
aff_stats = {
    "features": stage3_features,
    "mean": train_dataset_aff.cond_mean.tolist(),
    "std": train_dataset_aff.cond_std.tolist()
}
with open(os.path.join(config.MODEL_DIR, "vae_affinity_condition_stats.json"), "w") as f:
    json.dump(aff_stats, f, indent=2)

print("\nSaved Stage 3 condition stats -> vae_affinity_condition_stats.json")
print(f"Train batches (aff): {len(train_loader_aff)}")
print(f"Val batches (aff): {len(val_loader_aff)}")

In [ ]:
# Build Stage 3 CVAE (n_conditions changes => condition_net must be new)
vae_aff = MoleculeVAE(
    vocab_size=VOCAB_SIZE,
    embed_dim=config.EMBED_DIM,
    encoder_hidden=config.ENCODER_HIDDEN,
    encoder_layers=config.ENCODER_LAYERS,
    decoder_hidden=config.DECODER_HIDDEN,
    decoder_layers=config.DECODER_LAYERS,
    latent_dim=config.LATENT_DIM,
    dropout=config.DROPOUT,
    bidirectional=config.ENCODER_BIDIRECTIONAL,
    padding_idx=PAD_IDX,
    condition_dim=config.CONDITION_HIDDEN,
    n_conditions=len(stage3_features)
).to(device)

print(f"Stage 3 CVAE params: {count_parameters(vae_aff):,}")

# Load Stage 2 best model (encoder/decoder transfer)
stage2_ckpt_path = os.path.join(config.MODEL_DIR, "vae_cond_best.pt")
ckpt2 = torch.load(stage2_ckpt_path, map_location=device)

state2 = ckpt2["model_state_dict"]
state3 = vae_aff.state_dict()

# Copy matching keys (this will naturally skip condition_net if shapes differ)
loaded, skipped = [], []
for k, v in state2.items():
    if k in state3 and state3[k].shape == v.shape:
        state3[k] = v
        loaded.append(k)
    else:
        skipped.append(k)

vae_aff.load_state_dict(state3)

print(f"\nLoaded {len(loaded)} tensors from Stage 2 into Stage 3.")
print(f"Skipped {len(skipped)} tensors (expected: condition_net.* if n_conditions changed).")

# Stage 3 training hyperparams

In [ ]:
# Stage 3 training hyperparams
AFF_EPOCHS = 20
AFF_LR = 1e-4

optimizer_aff = AdamW(
    vae_aff.parameters(),
    lr=AFF_LR,
    weight_decay=config.WEIGHT_DECAY
)

scheduler_aff = ReduceLROnPlateau(
    optimizer_aff, mode="min", factor=0.5, patience=3, verbose=True
)

history_aff = defaultdict(list)
best_val_aff = float("inf")

print("\n" + "="*60)
print("TRAINING STAGE 3 (AFFINITY CVAE)")
print("="*60)

for epoch in range(AFF_EPOCHS):
    vae_aff.train()
    train_loss, train_recon, train_kl = train_vae_epoch(
        vae_aff, train_loader_aff, optimizer_aff, epoch, config, conditional=True
    )
    
    vae_aff.eval()
    val_loss, val_recon, val_kl = validate_vae(vae_aff, val_loader_aff, config, conditional=True)
    
    scheduler_aff.step(val_loss)
    
    history_aff["train_loss"].append(train_loss)
    history_aff["val_loss"].append(val_loss)
    history_aff["train_recon"].append(train_recon)
    history_aff["val_recon"].append(val_recon)
    history_aff["train_kl"].append(train_kl)
    history_aff["val_kl"].append(val_kl)
    
    print(f"Epoch {epoch+1}/{AFF_EPOCHS} | "
          f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
          f"Recon: {val_recon:.4f} | KL: {val_kl:.4f}")
    
    if val_loss < best_val_aff:
        best_val_aff = val_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": vae_aff.state_dict(),
            "optimizer_state_dict": optimizer_aff.state_dict(),
            "val_loss": val_loss,
            "history": dict(history_aff),
            "config": vars(config),
            "stage": 3,
            "n_conditions": len(stage3_features),
            "condition_features": stage3_features,
        }, os.path.join(config.MODEL_DIR, "vae_affinity_best.pt"))
        print(f"  ✓ Saved best Stage 3 model (val_loss: {val_loss:.4f})")

print("\n✅ Stage 3 training complete!")

In [ ]:
# Load best affinity model
ckpt_aff = torch.load(os.path.join(config.MODEL_DIR, "vae_affinity_best.pt"), map_location=device)
vae_aff.load_state_dict(ckpt_aff["model_state_dict"])
vae_aff.eval()
print(f"Loaded affinity model from epoch {ckpt_aff['epoch']+1}")

# Load Stage 3 condition stats
with open(os.path.join(config.MODEL_DIR, "vae_affinity_condition_stats.json"), "r") as f:
    aff_stats = json.load(f)

def prepare_conditions_from_stats(target_values, stats, num_samples, device):
    feats = stats["features"]
    mean = np.array(stats["mean"])
    std  = np.array(stats["std"])

    vals = [target_values.get(feat, mean[i]) for i, feat in enumerate(feats)]
    vals = np.array(vals)
    vals_norm = (vals - mean) / std

    return torch.tensor(vals_norm, dtype=torch.float32, device=device).unsqueeze(0).expand(num_samples, -1)

print("\n" + "="*60)
print("POTENCY-TARGETED SAMPLING (pchembl_mean target)")
print("="*60)

num_samples = 100

# Example: target pchembl_mean ~ 8.0 (very potent, ~10nM-ish scale depending on definition)
target_potent = {
    "qed": 0.80,
    "sa_score": 3.0,
    "pchembl_mean": 8.0
}

cond_batch = prepare_conditions_from_stats(target_potent, aff_stats, num_samples, device)
cond_emb = vae_aff.condition_net(cond_batch)

with torch.no_grad():
    z = torch.randn(num_samples, config.LATENT_DIM, device=device)
    samples = vae_aff.decode(
        z, config.MAX_LEN, cond_emb,
        temperature=config.SAMPLE_TEMP,
        top_p=config.TOP_P
    )

generated_selfies = [decode_tokens(s.cpu().tolist()) for s in samples]
generated_smiles  = [selfies_to_smiles(s) for s in generated_selfies]
valid_smiles      = [s for s in generated_smiles if is_valid_molecule(s)]

print(f"\nValidity: {len(valid_smiles)/len(generated_smiles)*100:.1f}%")
print("\nTop valid samples:")
for i, smi in enumerate(valid_smiles[:10]):
    print(f"  {i+1}. {smi}")

## Latent Space Interpolation

In [ ]:
# Interpolate between two molecules in latent space
print("\n" + "="*60)
print("LATENT SPACE INTERPOLATION")
print("="*60)

# Encode two molecules
mol1_selfies = df.iloc[0]['selfies']
mol2_selfies = df.iloc[100]['selfies']

mol1_tokens = torch.tensor([encode_selfies(mol1_selfies)], device=device)[:, :-1]
mol2_tokens = torch.tensor([encode_selfies(mol2_selfies)], device=device)[:, :-1]

vae_uncond.eval()
with torch.no_grad():
    mu1, _ = vae_uncond.encode(mol1_tokens)
    mu2, _ = vae_uncond.encode(mol2_tokens)

print(f"Molecule 1: {selfies_to_smiles(mol1_selfies)[:50]}...")
print(f"Molecule 2: {selfies_to_smiles(mol2_selfies)[:50]}...")

# Interpolate
print(f"\nInterpolation (10 steps):")
alphas = np.linspace(0, 1, 10)
interpolated_smiles = []

for alpha in alphas:
    z_interp = (1 - alpha) * mu1 + alpha * mu2
    with torch.no_grad():
        generated = vae_uncond.decode(
            z_interp, config.MAX_LEN, None,
            temperature=0.8, top_p=0.9
        )
    selfies = decode_tokens(generated[0].cpu().tolist())
    smiles = selfies_to_smiles(selfies)
    interpolated_smiles.append(smiles)
    valid = "✓" if is_valid_molecule(smiles) else "✗"
    print(f"  α={alpha:.1f}: {smiles[:40] if smiles else 'None'}... {valid}")

## Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 2, figsize=(14, 14))

# Row 1: Loss curves
axes[0, 0].plot(history_uncond['train_loss'], label='Train', alpha=0.8)
axes[0, 0].plot(history_uncond['val_loss'], label='Val', alpha=0.8)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Unconditional VAE - Total Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history_cond['train_loss'], label='Train', alpha=0.8)
axes[0, 1].plot(history_cond['val_loss'], label='Val', alpha=0.8)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Total Loss')
axes[0, 1].set_title('Conditional VAE - Total Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Row 2: Recon vs KL
axes[1, 0].plot(history_uncond['train_recon'], label='Recon', alpha=0.8)
axes[1, 0].plot(history_uncond['train_kl'], label='KL', alpha=0.8)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_title('Unconditional VAE - Recon vs KL')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history_cond['train_recon'], label='Recon', alpha=0.8)
axes[1, 1].plot(history_cond['train_kl'], label='KL', alpha=0.8)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Conditional VAE - Recon vs KL')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Row 3: Generation Quality Metrics
# Validity & Uniqueness
ax3 = axes[2, 0]
ax3.plot(history_uncond['validity'], label='Validity (Uncond)', marker='o', markersize=3)
ax3.plot(history_uncond['uniqueness'], label='Uniqueness (Uncond)', marker='s', markersize=3)
ax3.plot(history_cond['validity'], label='Validity (Cond)', marker='o', markersize=3, linestyle='--')
ax3.plot(history_cond['uniqueness'], label='Uniqueness (Cond)', marker='s', markersize=3, linestyle='--')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('Percentage (%)')
ax3.set_title('Generation Quality: Validity & Uniqueness')
ax3.legend(loc='lower right')
ax3.grid(True, alpha=0.3)
ax3.set_ylim(0, 105)

# Mean Length (degeneration detector)
ax4 = axes[2, 1]
ax4.plot(history_uncond['mean_length'], label='Uncond', marker='o', markersize=3)
ax4.plot(history_cond['mean_length'], label='Cond', marker='s', markersize=3)
ax4.fill_between(range(len(history_uncond['mean_length'])),
                  np.array(history_uncond['mean_length']) - np.array(history_uncond['std_length']),
                  np.array(history_uncond['mean_length']) + np.array(history_uncond['std_length']),
                  alpha=0.2)
ax4.axhline(y=10, color='red', linestyle='--', alpha=0.7, label='Degeneration threshold')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Mean Length (tokens)')
ax4.set_title('Mean Sequence Length (⚠️ <10 = EOS spam)')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.MODEL_DIR, 'vae_training_curves.png'), dpi=150)
plt.show()

## Summary

In [ ]:
print("="*70)
print("VAE TRAINING SUMMARY")
print("="*70)

print("\n📐 ARCHITECTURE:")
print(f"  Embedding: {config.EMBED_DIM}")
print(f"  Encoder: {config.ENCODER_LAYERS}x BiGRU-{config.ENCODER_HIDDEN}")
print(f"  Latent dimension: {config.LATENT_DIM}")
print(f"  Decoder: {config.DECODER_LAYERS}x GRU-{config.DECODER_HIDDEN}")
print(f"  Parameters: {count_parameters(vae_uncond):,} (uncond), {count_parameters(vae_cond):,} (cond), {count_parameters(vae_aff):,} (affinity)")

print("\n🎯 TRAINING STAGES:")
print(f"  Stage 1: Unconditional VAE ({config.EPOCHS} epochs, {len(train_df):,} molecules)")
print(f"  Stage 2: Conditional VAE ({config.EPOCHS} epochs, {len(train_df):,} molecules)")
print(f"  Stage 3: Affinity VAE ({AFF_EPOCHS} epochs, {len(df_aff):,} molecules)")
print(f"  Learning rates: {config.LEARNING_RATE} (S1/S2), {AFF_LR} (S3 fine-tune)")
print(f"  KL Annealing: {config.KL_ANNEAL_TYPE}")

print("\n📁 SAVED FILES:")
print(f"  ✅ {config.MODEL_DIR}/vae_uncond_best.pt")
print(f"  ✅ {config.MODEL_DIR}/vae_cond_best.pt")
print(f"  ✅ {config.MODEL_DIR}/vae_affinity_best.pt")
print(f"  ✅ {config.MODEL_DIR}/vae_condition_stats.json")
print(f"  ✅ {config.MODEL_DIR}/vae_affinity_condition_stats.json")
print(f"  ✅ {config.MODEL_DIR}/vae_training_curves.png")

print("\n🎯 CONDITIONING FEATURES:")
print("  Stage 2 (high coverage):")
for feat in available_features:
    print(f"    • {feat}")
print("  Stage 3 (+ affinity):")
for feat in stage3_features:
    if feat not in available_features:
        print(f"    • {feat} (binding affinity, pIC50)")

print("\n---")
print("STAGE 3 (AFFINITY CVAE):")
print("  • Fine-tune on pchembl_mean subset (~21% coverage)")
print(f"  • Epochs: {AFF_EPOCHS} (fine-tuning)")
print(f"  • LR: {AFF_LR}")
print("  • Init: encoder/decoder transferred from Stage 2; new condition_net")
print("  ✅ Saved: vae_affinity_best.pt")
print("  ✅ Saved: vae_affinity_condition_stats.json")

print("\n✅ VAE CAPABILITIES:")
print("  • Generate novel drug-like molecules (Stage 1)")
print("  • Condition on QED, SA, MW, alogP, etc. (Stage 2)")
print("  • Generate potent molecules with target affinity (Stage 3)")
print("  • Latent space interpolation")
print("  • Molecule optimization via latent search")

print("\n🚀 NEXT STEPS:")
print("  1. Use affinity VAE for potency-guided generation")
print("  2. Latent space optimization with RL/Bayesian methods")
print("  3. Compare VAE vs Diffusion model quality")
print("  4. Virtual screening with generated molecules")